## LIGHTGBM GBDT

In [ ]:
X1, X2, X3, X4, X5 = create_kfold_sets(data_train=data_train) #Model without feature selection
#X1, X2, X3, X4, X5 = create_kfold_sets(data_train=data_train_wrapper) #Model with feature selection

D = [X1, X2, X3, X4, X5]

In [ ]:
# Define the columns for the results DataFrame
columns = ['Model', 'N_estimators', 'Learning_rate','Num_leaves', 'Accuracy',
           'Recall',
           'Specificity',
           'Precision',
           'F1']
df_results = pd.DataFrame(columns=columns)

# Define parameters for Grid Search
n_estimators = [25, 50, 75, 100, 125, 150]
learning_rates = [0.025, 0.05, 0.1, 0.2, 0.4]
num_leaves = [10, 30, 50]
model_name = 'LightGBM GBDT'

for i in n_estimators:
    for d in learning_rates:
        for l in num_leaves:
          df_results_fold = pd.DataFrame(columns=['Model', 'Accuracy', 'Recall', 'Specificity', 'Precision', 'F1'])
              for j in range(5):
                  # Prepare test and train sets for this fold
                  d_test = pd.concat([D[j]])
                  y_test = d_test['label_binary']
                  X_test = d_test.drop(columns=['label_binary', 'n_image', 'label_multi'])
                  d_train = pd.concat([D[k] for k in range(5) if k != j], ignore_index=True)
                  y_train = d_train['label_binary']
                  X_train = d_train.drop(columns=['label_binary', 'n_image', 'label_multi'])
                  label_mapping = {'no corrosion': 0, 'corrosion': 1}
                  y_train = y_train.map(label_mapping)
                  y_test = y_test.map(label_mapping)

                  # Initialize and train the LightGBM model
                  model = lgbm(
                      objective='binary',  # Binary classification
                      boosting_type='gbdt',  # Boosting type
                      n_estimators=i,             # Number of trees (boosting rounds)
                      seed=42,                       # For reproducibility
                      learning_rate=d,
                      n_bins=k,
                      num_leaves=l,
                      eval_metric=m,
                      verbose=-1
                  )
                  model.fit(X_train, y_train)

                  y_pred = model.predict(X_test)
                  print(f'Model with {i} estimators, {d} learning rate, and {l} leaves:')
                  labels = (1, 0)
                  cm = confusion_matrix(y_test, y_pred, labels=labels)
                  print(f'Confusion matrix for {i} estimators: \n {cm}')
                  results=evaluate_model(y_pred, y_test, model=model_name, labels=(1,0))
                  df_results_fold = pd.concat([df_results_fold, results], ignore_index=True)
                  print('\n')



              # Calculate mean metrics across folds
              recall_mean, specificity_mean, precision_mean, f1_mean, accuracy_mean = means_results(df_results_fold)


              # Append results to DataFrame
              result_i = {'Model': model_name, 'Accuracy': accuracy_mean, 'N_estimators': i, 'Learning_rate': d, 'Num_leaves': l,
                          'Recall': recall_mean, 'Specificity': specificity_mean,
                          'Precision': precision_mean, 'F1': f1_mean,}
              df_results = pd.concat([df_results, pd.DataFrame([result_i])], ignore_index=True)


In [ ]:
df_results.sort_values(by='Recall', ascending=False)

Chosen model

In [ ]:
# Parameters
n_estimators = 25
learning_rate = 0.025
num_leaves = 10
model_name = 'LightGBM GBDT'

# Prepare data
X_train = data_train.drop(columns=['label_binary', 'n_image', 'label_multi'])
y_train = data_train['label_binary']
label_mapping = {'no corrosion': 0, 'corrosion': 1}
y_train = y_train.map(label_mapping)
X_test = data_test.drop(columns=['label_binary', 'n_image', 'label_multi'])
y_test = data_test['label_binary']
y_test = y_test.map(label_mapping)

# Initialize and train the LightGBM model
model = lgbm(
    objective='binary',  # Binary classification
    boosting_type='gbdt',  # Boosting type
    n_estimators=n_estimators,             # Number of trees (boosting rounds)
    seed=42,                       # For reproducibility
    learning_rate=learning_rate,
    eval_metric='logloss',
    verbose=-1,
    num_leaves=num_leaves
)
model.fit(X_train, y_train)

# Measure execution time
t0 = time.time()
y_pred = model.predict(X_test)
t1 = time.time()
time_taken = t1 - t0
time_taken = round(time_taken, 3)

recall, specificity, precision, f1, accuracy = evaluate_model(y_pred, y_test, model='LightGBM GBDT')

# Print results
print(f'Execution time: {time_taken} seconds')

# Create a DataFrame to store results
columns = ['Model', 'N_estimators', 'Accuracy', 'Learning_rate', 'Num_leaves',
           'Metric',
           'Recall',
           'Specificity',
           'Precision',
           'F1',
           'Time']
df_results = pd.DataFrame(columns=columns)

# Append results to DataFrame
result_i = {'Model': model_name, 'N_estimators': n_estimators, 'Learning_rate': learning_rate, 'Num_leaves': num_leaves,
            'Accuracy': accuracy,
            'Recall': recall, 'Specificity': specificity,
            'Precision': precision, 'F1': f1,
            'Time': time_taken}
df_results = pd.concat([df_results, pd.DataFrame([result_i])], ignore_index=True)

print(df_results)
